In [2]:
import cv2
import numpy as np
import pandas as pd

# Entropy Function

In [3]:
def my_entropy(input_image):

    arr = input_image.flatten().astype(np.int64)

    if arr.min() != 1:
        arr = arr - arr.min() + 1

    p = np.zeros(arr.max(), dtype=np.float64)
    for v in arr:
        p[v - 1] += 1

    p = p / p.sum()
    p = p[p != 0]
    entropy = np.sum(-p * np.log2(p))

    return entropy

## PSNR Function

In [4]:
def peak_signal_noise_ratio(image1: np.ndarray, image2: np.ndarray, max_value=255):
    if image1.shape != image2.shape:
        print('Input images don’t have the same shape')
        return -1

    max_value = max_value
    max_value_square = max_value ** 2

    # mean-square-error
    img1_flat = image1.flatten()
    img2_flat = image2.flatten()

    error = img1_flat - img2_flat
    error_square = error ** 2
    error_square_sum = np.sum(error_square)
    mean_error_square_sum = error_square_sum / len(img1_flat)

    if mean_error_square_sum == 0:
        return np.inf
    return 10 * np.log10(max_value_square / (mean_error_square_sum))

# Median Edge Predictor (MED)

In [5]:
def med_predictor(input_image):
    input_image = input_image.astype(np.int16)
    H, W = input_image.shape

    error_image = np.zeros((H, W), dtype=np.int16)

    padded_Image = np.pad(input_image, ((1, 0), (1, 0)), mode='constant', constant_values=0)

    for i in range(1, H + 1):
        for j in range(1, W + 1):

            a = padded_Image[i, j - 1]
            b = padded_Image[i - 1, j]
            c = padded_Image[i - 1, j - 1]

            if c >= max(a, b):
                x = min(a, b)
            elif c <= min(a, b):
                x = max(a, b)
            else:
                x = a + b - c

            error_image[i - 1, j - 1] = input_image[i - 1, j - 1] - x

    return error_image


def med_reconstructor(error_image):
    error_image = error_image.astype(np.int16)
    H, W = error_image.shape

    prediction = np.zeros((H + 1, W + 1), dtype=np.int16)
    reconstructed_image = np.zeros((H, W), dtype=np.uint8)

    for i in range(1, H + 1):
        for j in range(1, W + 1):
            a = prediction[i, j - 1]
            b = prediction[i - 1, j]
            c = prediction[i - 1, j - 1]

            if c >= max(a, b):
                x = min(a, b)
            elif c <= min(a, b):
                x = max(a, b)
            else:
                x = a + b - c

            prediction[i, j] = x + error_image[i - 1, j - 1]
            reconstructed_image[i - 1, j - 1] = np.uint8(prediction[i, j])

    return reconstructed_image

## MED Evaluation

In [6]:
test_path = ['Images/CLIC_2025_1.png', 'Images/CLIC_2025_2.png', 'Images/Kodak_01.png', 'Images/Kodak_23.png',
             'Images/Livingroom.tif', 'Images/Bridge.tif', 'Images/Baboon.tif', 'Images/Peppers.bmp',
             'Images/MRI_1.tif', 'Images/MRI_2.tif', 'Images/Retina.tif', 'Images/Cells.png']

evaluation_table = []
for image_name in test_path:
    name = image_name.split('/')[1].split('.')[0]
    image = cv2.imread(image_name, cv2.IMREAD_UNCHANGED)

    med_error = med_predictor(image)
    med_reconstruct = med_reconstructor(med_error)

    evaluation_table.append({'Image Name': name,'Image Entropy': my_entropy(image),
                             'Error Entropy(MED)': my_entropy(med_error), 'PSNR (initial-reconstruct)': peak_signal_noise_ratio(image, med_reconstruct)})

entropy_table = pd.DataFrame(evaluation_table)
entropy_table

,Image Name,Image Entropy,Error Entropy(MED),PSNR (initial-reconstruct)
0,CLIC_2025_1,7.573678,4.024232,inf
1,CLIC_2025_2,7.615759,5.889772,inf
2,Kodak_01,7.161006,5.511179,inf
3,Kodak_23,7.251587,3.829204,inf
4,Livingroom,7.295174,4.839180,inf
5,Bridge,7.683018,5.668946,inf
6,Baboon,7.292549,5.233969,inf
7,Peppers,7.571478,4.843694,inf
8,MRI_1,6.428016,3.765981,inf
9,MRI_2,6.619796,3.612304,inf
